**Construct SR Model**
- You need to build the ZebraSRNet by completing the class definition of **ResBlock_typeB**, **upsampler** and **ZebraSRNet**.

In [ ]:
import torch
import torch.nn as nn
from types import SimpleNamespace
import os
import numpy
import random

- Example model construction in pytorch for Residual Block TypeA.

In [ ]:
'''
    Example model construction in pytorch
'''
class ResBlock_typeA(nn.Module):
  def __init__(self, nFeat):
    super(ResBlock_typeA, self).__init__()
    modules = []
    modules.append(nn.Conv2d(nFeat, nFeat, 3, padding=1, bias=True))  # 3x3 conv
    modules.append(nn.SiLU(True))       # SiLU
    #modules.append(nn.GELU())          # GeLU
    modules.append(nn.Conv2d(nFeat, nFeat, 3, padding=1, bias=True))  # 3x3 conv
    self.body = nn.Sequential(*modules)

  def forward(self, x):
    out = self.body(x)
    out += x
    return out


- The following is your implementation of the model construction.

In [ ]:
class ResBlock_typeB(nn.Module):
  def __init__(self, nFeat, ReduceRatio=2, bias=True):
    super(ResBlock_typeB, self).__init__()
    #===== write your model definition here =====#

    modules = []
    modules.append(nn.Conv2d(nFeat, round(nFeat / ReduceRatio), 1, bias=True))  # 1x1 conv
    modules.append(nn.SiLU(True))       # SiLU
    #modules.append(nn.GELU())          # GeLU
    modules.append(nn.Conv2d(round(nFeat / ReduceRatio), round(nFeat / ReduceRatio), 3, padding=1, bias=True))  # 3x3 conv
    modules.append(nn.SiLU(True))       # SiLU
    #modules.append(nn.GELU())
    modules.append(nn.Conv2d(round(nFeat / ReduceRatio), nFeat, 1, bias=True))  # 1x1 conv
    self.body = nn.Sequential(*modules)

    #============================================#

  def forward(self, x):
    #===== write your dataflow here =====#

    out = self.body(x)
    out += x

    #====================================#
    return out

In [ ]:
class upsampler(nn.Module):
  def __init__(self, nFeat, scale=2):
    super(upsampler, self).__init__()
    #===== write your model definition here =====#

    modules = []
    modules.append(nn.Conv2d(nFeat, nFeat * scale * scale, 3, padding=1, bias=True))
    modules.append(nn.PixelShuffle(scale))
    modules.append(nn.GELU())
    self.body = nn.Sequential(*modules)

    #============================================#

  def forward(self, x):
    #===== write your dataflow here =====#

    out = self.body(x)

    #====================================#
    return out

In [ ]:
class ZebraSRNet(nn.Module):
  def __init__(self, nFeat=64, ReduceRatio=4, nResBlock=8, imgChannel=3):
    super(ZebraSRNet, self).__init__()
    #===== write your model definition here, using 'ResBlock_typeB' and 'upsampler' as the building blocks =====#
    #===== Note that the number of residual blocks should change according to nResBlock. =======================#

    self.head = nn.Conv2d(imgChannel, nFeat, 3, padding=1, bias=True)

    body = [ResBlock_typeB(nFeat, ReduceRatio, bias=True) for _ in range(nResBlock)]
    self.body = nn.Sequential(*body)

    self.up1 = upsampler(nFeat)
    self.up2 = upsampler(nFeat)

    self.tail = nn.Conv2d(nFeat, imgChannel, 3, padding=1, bias=True)

    #============================================================================================================#


  def forward(self, x):
    #===== write your dataflow here =====#

    out = self.head(x)
    out = out + self.body(out)
    out = self.up1(out)
    out = self.up2(out)
    out = self.tail(out)

    #====================================#
    return out